In [0]:
ratings_data = [

    (101, 1, 5, "2024-01-01"),
    (102, 1, 4, "2024-01-02"),
    (103, 2, 3, "2024-01-03"),
    (104, 3, 5, "2024-01-04"),
    (105, 2, 2, "2024-01-05"),
    (106, 4, 4, "2024-01-06"),
    (107, 5, 5, "2024-01-07"),
    (108, 3, 1, "2024-01-08"),
    (109, 4, 3, "2024-01-09"),
    (110, 5, 4, "2024-01-10"),

    # bad records
    (111, 6, None, "2024-01-11"),
    (112, 7, 10, "2024-01-12"),
    (113, None, 4, "2024-01-13")

]

ratings_columns = [
    "user_id",
    "movie_id",
    "rating",
    "rating_date"
]

ratings_df = spark.createDataFrame(
    ratings_data,
    ratings_columns
)

display(ratings_df)

user_id,movie_id,rating,rating_date
101,1,5,2024-01-01
102,1,4,2024-01-02
103,2,3,2024-01-03
104,3,5,2024-01-04
105,2,2,2024-01-05
106,4,4,2024-01-06
107,5,5,2024-01-07
108,3,1,2024-01-08
109,4,3,2024-01-09
110,5,4,2024-01-10


In [0]:
movies_data = [

    (1, "Inception", "Sci-Fi", 2010),
    (2, "Titanic", "Romance", 1997),
    (3, "Avengers", "Action", 2012),
    (4, "Interstellar", "Sci-Fi", 2014),
    (5, "Joker", "Drama", 2019)

]

movies_columns = [
    "movie_id",
    "title",
    "genre",
    "release_year"
]

movies_df = spark.createDataFrame(
    movies_data,
    movies_columns
)

display(movies_df)

movie_id,title,genre,release_year
1,Inception,Sci-Fi,2010
2,Titanic,Romance,1997
3,Avengers,Action,2012
4,Interstellar,Sci-Fi,2014
5,Joker,Drama,2019


In [0]:
#detecting corrupt records
bad_records = ratings_df.filter(

    (ratings_df.rating.isNull()) |
    (ratings_df.movie_id.isNull()) |
    (ratings_df.rating < 1) |
    (ratings_df.rating > 5)

)

display(bad_records)

user_id,movie_id,rating,rating_date
111,6,null,2024-01-11
112,7,10,2024-01-12
113,null,4,2024-01-13


In [0]:
#cleandatasets
ratings_clean = ratings_df.dropna(
    subset=["movie_id", "rating"]
)

ratings_clean = ratings_clean.dropDuplicates()

ratings_clean = ratings_clean.filter(
    (ratings_clean.rating >= 1) &
    (ratings_clean.rating <= 5)
)

display(ratings_clean)

user_id,movie_id,rating,rating_date
101,1,5,2024-01-01
102,1,4,2024-01-02
103,2,3,2024-01-03
104,3,5,2024-01-04
105,2,2,2024-01-05
106,4,4,2024-01-06
107,5,5,2024-01-07
108,3,1,2024-01-08
109,4,3,2024-01-09
110,5,4,2024-01-10


In [0]:
#join ratings with movie table
movie_ratings = ratings_clean.join(
    movies_df,
    on="movie_id",
    how="inner"
)

display(movie_ratings)

movie_id,user_id,rating,rating_date,title,genre,release_year
1,101,5,2024-01-01,Inception,Sci-Fi,2010
1,102,4,2024-01-02,Inception,Sci-Fi,2010
2,103,3,2024-01-03,Titanic,Romance,1997
2,105,2,2024-01-05,Titanic,Romance,1997
3,104,5,2024-01-04,Avengers,Action,2012
4,106,4,2024-01-06,Interstellar,Sci-Fi,2014
5,107,5,2024-01-07,Joker,Drama,2019
3,108,1,2024-01-08,Avengers,Action,2012
5,110,4,2024-01-10,Joker,Drama,2019
4,109,3,2024-01-09,Interstellar,Sci-Fi,2014


In [0]:
#average rating per movie
from pyspark.sql.functions import avg, round

avg_movie_rating = movie_ratings.groupBy(
    "movie_id",
    "title"
).agg(
    round(avg("rating"), 2).alias("avg_rating")
)

display(avg_movie_rating)

movie_id,title,avg_rating
1,Inception,4.5
2,Titanic,2.5
3,Avengers,3.0
4,Interstellar,3.5
5,Joker,4.5


In [0]:
#popular movies
from pyspark.sql.functions import count

popular_movies = movie_ratings.groupBy(
    "movie_id",
    "title"
).agg(
    count("rating").alias("total_ratings")
).orderBy(
    "total_ratings",
    ascending=False
)

display(popular_movies)

movie_id,title,total_ratings
1,Inception,2
2,Titanic,2
3,Avengers,2
4,Interstellar,2
5,Joker,2


In [0]:
#user behaviour analytics
user_activity = movie_ratings.groupBy(
    "user_id"
).agg(
    count("movie_id").alias("movies_rated")
).orderBy(
    "movies_rated",
    ascending=False
)

display(user_activity)

user_id,movies_rated
101,1
102,1
103,1
105,1
104,1
106,1
107,1
108,1
110,1
109,1


In [0]:
#genre analytics
genre_analytics = movie_ratings.groupBy(
    "genre"
).agg(
    round(avg("rating"), 2).alias("avg_genre_rating"),
    count("movie_id").alias("total_ratings")
)

display(genre_analytics)

genre,avg_genre_rating,total_ratings
Sci-Fi,4.0,4
Romance,2.5,2
Action,3.0,2
Drama,4.5,2


In [0]:
#trending movie
from pyspark.sql.functions import current_date, datediff

trending_movies = movie_ratings.filter(
    datediff(current_date(), movie_ratings.rating_date) <=3
    ).groupBy(
    "movie_id",
    "title"
).agg(
    count("rating").alias("recent_ratings")
).orderBy(
    "recent_ratings",
    ascending=False
)

display(trending_movies)

movie_id,title,recent_ratings


In [0]:
#rank movies using window function
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

window_spec = Window.orderBy(
    avg_movie_rating["avg_rating"].desc()
)

ranked_movies = avg_movie_rating.withColumn(
    "movie_rank",
    rank().over(window_spec)
)

display(ranked_movies)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


movie_id,title,avg_rating,movie_rank
1,Inception,4.5,1
5,Joker,4.5,1
4,Interstellar,3.5,3
3,Avengers,3.0,4
2,Titanic,2.5,5


In [0]:
#incremental load
new_ratings_data = [

    (120, 1, 5, "2024-02-01"),
    (121, 2, 4, "2024-02-01")

]

new_ratings_df = spark.createDataFrame(
    new_ratings_data,
    ratings_columns
)

display(new_ratings_df)

user_id,movie_id,rating,rating_date
120,1,5,2024-02-01
121,2,4,2024-02-01


In [0]:
#append incremental data
final_ratings = ratings_clean.union(
    new_ratings_df
)

display(final_ratings)

user_id,movie_id,rating,rating_date
101,1,5,2024-01-01
102,1,4,2024-01-02
103,2,3,2024-01-03
105,2,2,2024-01-05
104,3,5,2024-01-04
106,4,4,2024-01-06
107,5,5,2024-01-07
108,3,1,2024-01-08
110,5,4,2024-01-10
109,4,3,2024-01-09
